# Market Basket Analysis on Retail Data

**Goal:** find products that customers tend to buy together, so a store could use them for cross-selling.

**Before you run:** download the **Online Retail II** dataset (UCI Machine Learning Repository, also on Kaggle) or the older **Online Retail** dataset from Kaggle. Put the file (`.xlsx` or `.csv`) in the same folder as this notebook, or upload it in Colab. The notebook finds it automatically if the file name contains "online_retail" (any capitalisation).

**Plan:** clean the data in pandas, load it into SQLite, use SQL to build the baskets, then run the Apriori algorithm to get association rules.

Key terms (you'll see them in the output):
- **Support:** share of baskets that contain the item(s).
- **Confidence:** of the baskets with item A, the share that also have item B.
- **Lift:** how much more often A and B appear together than if they were unrelated. Lift above 1 means a positive association.

## 1. Imports

In [ ]:
import glob, sqlite3, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import mlxtend
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mlxtend", "openpyxl"])
    import mlxtend

from mlxtend.frequent_patterns import apriori, association_rules
print("mlxtend version:", mlxtend.__version__)

## 2. Load the data

In [ ]:
matches = [f for f in glob.glob("*") if "online_retail" in f.lower().replace(" ", "_")
           and f.lower().endswith((".xlsx", ".xls", ".csv"))]
print("Found:", matches)
FILE = matches[0]   # change this line if you want a specific file

if FILE.lower().endswith((".xlsx", ".xls")):
    # Online Retail II has two sheets (2009-2010 and 2010-2011): read and join them
    sheets = pd.read_excel(FILE, sheet_name=None)
    df = pd.concat(sheets.values(), ignore_index=True)
else:
    df = pd.read_csv(FILE, encoding="ISO-8859-1")

# the two versions of the dataset use slightly different column names
df = df.rename(columns={"InvoiceNo": "Invoice", "UnitPrice": "Price", "CustomerID": "Customer ID"})
print("Shape:", df.shape)
df.head()

## 3. Clean the data
- Invoices starting with `C` are cancellations/returns, so remove them.
- Remove rows with missing description, zero or negative quantity, or zero price.
- Keep only real products (stock codes that start with 5 digits). Codes like `POST`, `M`, `BANK CHARGES` are fees, not products.

In [ ]:
print("Rows before cleaning:", len(df))
print("Missing values per column:\n", df.isnull().sum(), "\n")

df["Invoice"] = df["Invoice"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)

df = df[~df["Invoice"].str.startswith("C")]
df = df.dropna(subset=["Description"])
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
df = df[df["StockCode"].str.match(r"^\d{5}")]
df["Description"] = df["Description"].str.strip().str.upper()

print("Rows after cleaning:", len(df))
print("Invoices:", df["Invoice"].nunique(), "| Products:", df["Description"].nunique())
print("\nTop countries by number of invoices:")
print(df.groupby("Country")["Invoice"].nunique().sort_values(ascending=False).head(5))

## 4. Load into SQLite and explore with SQL
We use only one country so the baskets are comparable. The United Kingdom has by far the most invoices; if it's missing we fall back to all rows.

In [ ]:
COUNTRY = "United Kingdom"
sales = df[df["Country"] == COUNTRY] if (df["Country"] == COUNTRY).any() else df
sales = sales[["Invoice", "StockCode", "Description", "Quantity", "Price"]]

conn = sqlite3.connect(":memory:")
sales.to_sql("sales", conn, index=False, if_exists="replace")
print("Rows in SQL table:", pd.read_sql("SELECT COUNT(*) AS n FROM sales", conn)["n"][0])

In [ ]:
# Top 10 products by the number of baskets they appear in, ranked with a window function
q_top = '''
SELECT Description,
       COUNT(DISTINCT Invoice) AS baskets,
       RANK() OVER (ORDER BY COUNT(DISTINCT Invoice) DESC) AS popularity_rank
FROM sales
GROUP BY Description
ORDER BY baskets DESC
LIMIT 10
'''
pd.read_sql(q_top, conn)

In [ ]:
# Basket size: how many different items does a typical invoice have?
q_size = '''
SELECT items_in_basket, COUNT(*) AS num_baskets
FROM (
    SELECT Invoice, COUNT(DISTINCT Description) AS items_in_basket
    FROM sales
    GROUP BY Invoice
)
GROUP BY items_in_basket
ORDER BY items_in_basket
'''
size_df = pd.read_sql(q_size, conn)
print(size_df.head(10))
total = size_df["num_baskets"].sum()
print("\nBaskets with only 1 item: {:.1f}%".format(100 * size_df.loc[size_df.items_in_basket == 1, "num_baskets"].sum() / total))

## 5. Build the baskets with SQL
To keep Apriori fast, we keep the 200 most common products. A CTE picks them, then a join keeps only those rows. Baskets with fewer than 2 items can't form a pair, so we drop them.

In [ ]:
TOP_N = 200

q_basket = f'''
WITH top_items AS (
    SELECT Description
    FROM sales
    GROUP BY Description
    ORDER BY COUNT(DISTINCT Invoice) DESC
    LIMIT {TOP_N}
),
basket_items AS (
    SELECT DISTINCT s.Invoice, s.Description
    FROM sales s
    JOIN top_items t ON s.Description = t.Description
),
multi_item AS (
    SELECT Invoice FROM basket_items GROUP BY Invoice HAVING COUNT(*) >= 2
)
SELECT b.Invoice, b.Description
FROM basket_items b
JOIN multi_item m ON b.Invoice = m.Invoice
'''
basket_long = pd.read_sql(q_basket, conn)
print("Rows:", len(basket_long), "| baskets:", basket_long["Invoice"].nunique(), "| items:", basket_long["Description"].nunique())

In [ ]:
# Turn into a True/False table: one row per basket, one column per product
basket = pd.crosstab(basket_long["Invoice"], basket_long["Description"]) > 0
print("Basket matrix shape:", basket.shape)

## 6. Apriori: find frequent item sets
`MIN_SUPPORT = 0.02` means we only look at item combinations present in at least 2% of baskets. If the result is empty, lower it (e.g. 0.015). If it takes too long, raise it.

In [ ]:
MIN_SUPPORT = 0.02

freq = apriori(basket, min_support=MIN_SUPPORT, use_colnames=True, max_len=3)
freq["size"] = freq["itemsets"].apply(len)
print("Frequent item sets found:", len(freq))
print(freq["size"].value_counts().sort_index())

## 7. Association rules

In [ ]:
try:
    rules = association_rules(freq, metric="lift", min_threshold=1.0)
except TypeError:
    # newer mlxtend versions also want the number of baskets
    rules = association_rules(freq, num_itemsets=len(basket), metric="lift", min_threshold=1.0)

def to_text(s):
    return ", ".join(sorted(s))

rules["antecedents"] = rules["antecedents"].apply(to_text)
rules["consequents"] = rules["consequents"].apply(to_text)
rules = rules[["antecedents", "consequents", "support", "confidence", "lift"]]

print("Total rules with lift > 1:", len(rules))
print("Rules with lift >= 3:", int((rules["lift"] >= 3).sum()))
print("Rules with confidence >= 0.5:", int((rules["confidence"] >= 0.5).sum()))

In [ ]:
# Strong rules: confidence >= 0.5, sorted by lift
strong = rules[rules["confidence"] >= 0.5].sort_values("lift", ascending=False)
strong.head(15).round(3)

## 8. Plot
Each dot is a rule. The best rules are towards the top right (high support and confidence) and darker (high lift).

In [ ]:
plt.figure(figsize=(7, 5))
sc = plt.scatter(rules["support"], rules["confidence"], c=rules["lift"], cmap="viridis", alpha=0.7)
plt.colorbar(sc, label="Lift")
plt.xlabel("Support")
plt.ylabel("Confidence")
plt.title("Association rules")
plt.grid(alpha=0.3)
plt.show()

**Write your own cross-sell suggestions here (2-3).** Pick rules from the table above that make sense to a person, for example two products from the same set. For each one, say what a store manager could do (put them next to each other, bundle them, recommend one when the other is in the cart).

## 9. Copy this block and send it back

In [ ]:
print("=== SUMMARY ===")
print("Dataset file:", FILE)
print("Rows after cleaning:", len(df))
print("Country used:", COUNTRY if (df["Country"] == COUNTRY).any() else "all")
print("Invoices (all):", sales["Invoice"].nunique())
print("Baskets used (2+ items, top", TOP_N, "products):", basket.shape[0])
print("Min support:", MIN_SUPPORT)
print("Frequent item sets:", len(freq))
print("Total rules (lift > 1):", len(rules))
print("Rules with lift >= 3:", int((rules["lift"] >= 3).sum()))
print("Rules with confidence >= 0.5:", int((rules["confidence"] >= 0.5).sum()))
print()
print("Top 10 strong rules (confidence >= 0.5, by lift):")
print(strong.head(10).round(3).to_string(index=False))